<a href="https://colab.research.google.com/github/vince-fabmob/open_data_extractions/blob/main/bixi_saint_leonard_extraction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Extraction BIXI — Saint-Léonard

Notebook Google Colab pour extraire les données ouvertes BIXI (historique des déplacements et état des stations via GBFS), isoler les stations situées dans l'arrondissement de Saint-Léonard (et à proximité immédiate), puis produire des agrégats de base (station, mois, heure, origine-destination).

**Sources :**
- Historique des déplacements BIXI (Ville de Montréal / BIXI)
- Flux GBFS `station_information` et `station_status` (BIXI Montréal)

**Prérequis Colab :**
- Téléverser un fichier `saint_leonard_limite.geojson` (limite administrative de l'arrondissement) dans l'environnement Colab, ou le rendre accessible via Google Drive.


## 1. Installation et répertoires

In [2]:
!pip -q install pandas geopandas shapely requests beautifulsoup4 pyarrow folium

from pathlib import Path
import io
import re
import zipfile
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import requests
from bs4 import BeautifulSoup

import geopandas as gpd

# Facultatif : décommentez pour conserver les données dans Google Drive.
# from google.colab import drive
# drive.mount("/content/drive")
# BASE_DIR = Path("/content/drive/MyDrive/analyse_mobilite/bixi_saint_leonard")
BASE_DIR = Path("/content/bixi_saint_leonard")

RAW_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"
OUTPUT_DIR = BASE_DIR / "outputs"
for directory in [RAW_DIR, PROCESSED_DIR, OUTPUT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

YEARS = [2021, 2022, 2023, 2024, 2025, 2026]
BUFFER_METERS = 500

## 2. Stations BIXI (GBFS)

In [3]:
GBFS_DISCOVERY_URL = "https://gbfs.velobixi.com/gbfs/gbfs.json"

def get_gbfs_feeds(discovery_url=GBFS_DISCOVERY_URL):
    response = requests.get(discovery_url, timeout=30)
    response.raise_for_status()
    payload = response.json()

    feeds = {}
    for language_block in payload.get("data", {}).values():
        for feed in language_block.get("feeds", []):
            feeds[feed["name"]] = feed["url"]
    return feeds

def fetch_gbfs_json(url):
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    return response.json()

feeds = get_gbfs_feeds()

station_information_url = feeds["station_information"]
station_status_url = feeds["station_status"]

station_info = fetch_gbfs_json(station_information_url)
station_status = fetch_gbfs_json(station_status_url)

stations = pd.DataFrame(station_info["data"]["stations"])
status = pd.DataFrame(station_status["data"]["stations"])

stations.columns = [c.lower() for c in stations.columns]
status.columns = [c.lower() for c in status.columns]

stations = stations.rename(columns={"station_id": "station_code"})
status = status.rename(columns={"station_id": "station_code"})

stations["station_code"] = stations["station_code"].astype(str)
status["station_code"] = status["station_code"].astype(str)

stations_current = stations.merge(status, on="station_code", how="left", suffixes=("", "_status"))
stations_current["extracted_at_utc"] = datetime.now(timezone.utc).isoformat()
stations_current.to_parquet(PROCESSED_DIR / "bixi_stations_current.parquet", index=False)

stations_current[["station_code", "name", "lat", "lon", "capacity", "num_bikes_available", "num_docks_available"]].head()

,station_code,name,lat,lon,capacity,num_bikes_available,num_docks_available
0,1,Drummond / de Maisonneuve,45.499655,-73.576335,23,1,21
1,2,Ste-Catherine / Dézéry,45.539230,-73.541088,27,6,21
2,3,Clark / Evans,45.511087,-73.567849,19,17,0
3,4,du Champ-de-Mars / Gosford,45.509655,-73.554009,23,4,19
4,5,Brittany / Ainsley,45.525890,-73.650034,22,7,14


## 3. Sélection territoriale — Saint-Léonard

In [6]:
# Étape 3 — Extraire la limite de Saint-Léonard de la couche administrative complète

from pathlib import Path
import unicodedata
import geopandas as gpd

SOURCE_FILE = Path(
    "/content/limites-administratives-agglomeration-nad83.geojson"
)

if not SOURCE_FILE.exists():
    raise FileNotFoundError(
        f"Fichier introuvable : {SOURCE_FILE}. "
        "Téléversez à nouveau le GeoJSON dans Colab, puis relancez la cellule."
    )

boundaries = gpd.read_file(SOURCE_FILE)

print("Système de coordonnées source :", boundaries.crs)
print("Colonnes disponibles :", list(boundaries.columns))

# Cherche une colonne qui contient le nom de l'arrondissement ou de la ville liée.
name_candidates = ["NOM", "nom", "NAME", "name", "NOM_ARR", "ARRONDISSEMENT"]
name_column = next(
    (column for column in name_candidates if column in boundaries.columns),
    None
)

if name_column is None:
    raise KeyError(
        "Impossible d'identifier une colonne de nom territorial. "
        f"Colonnes disponibles : {list(boundaries.columns)}"
    )

def normalize_text(value):
    value = str(value)
    value = unicodedata.normalize("NFKD", value)
    value = "".join(char for char in value if not unicodedata.combining(char))
    return value.lower().strip()

normalized_names = boundaries[name_column].map(normalize_text)

saint_leonard = boundaries[
    normalized_names.str.contains(
        r"saint[- ]?leonard",
        regex=True,
        na=False
    )
].copy()

if saint_leonard.empty:
    print("Aucune correspondance automatique. Valeurs disponibles :")
    display(
        boundaries[[name_column]]
        .drop_duplicates()
        .sort_values(name_column)
        .reset_index(drop=True)
    )
    raise ValueError(
        "Saint-Léonard n'a pas été trouvé. Vérifiez la colonne et les valeurs affichées."
    )

# Les limites sont en NAD83 : reprojection explicite en WGS 84 pour GBFS et Folium.
saint_leonard = saint_leonard.to_crs("EPSG:4326")

BOROUGH_FILE = Path("/content/saint_leonard_limite.geojson")
saint_leonard.to_file(BOROUGH_FILE, driver="GeoJSON")

print(f"Limite créée : {BOROUGH_FILE}")
print(f"Entité(s) retenue(s) : {len(saint_leonard)}")
display(saint_leonard.drop(columns="geometry"))

Système de coordonnées source : EPSG:32188
Colonnes disponibles : ['CODEID', 'NOM', 'NOM_OFFICIEL', 'CODEMAMH', 'CODE_3C', 'NUM', 'ABREV', 'TYPE', 'COMMENT', 'DATEMODIF', 'geometry']
Limite créée : /content/saint_leonard_limite.geojson
Entité(s) retenue(s) : 1


,CODEID,NOM,NOM_OFFICIEL,CODEMAMH,CODE_3C,NUM,ABREV,TYPE,COMMENT,DATEMODIF
19,6,Saint-Léonard,Saint-Léonard,REM14,STL,14,LN,Arrondissement,None,2023-11-29


## 4. Découverte des archives historiques BIXI

In [9]:
OPEN_DATA_URL = "https://bixi.com/fr/donnees-ouvertes"

# Étape 4 — Découvrir les archives BIXI depuis l'API CKAN de Montréal

import re
import pandas as pd
import requests

CKAN_BASE_URL = "https://donnees.montreal.ca/api/3/action"
DATASET_ID = "bixi-historique-des-deplacements"

def ckan_action(action, **params):
    response = requests.get(
        f"{CKAN_BASE_URL}/{action}",
        params=params,
        headers={"User-Agent": "Mozilla/5.0"},
        timeout=60,
    )
    response.raise_for_status()

    payload = response.json()

    if not payload.get("success"):
        raise RuntimeError(
            f"Erreur CKAN pour {action}: {payload.get('error')}"
        )

    return payload["result"]

dataset = ckan_action("package_show", id=DATASET_ID)

resources = pd.DataFrame(dataset["resources"])

# Colonnes utiles seulement, en conservant celles réellement offertes par CKAN
display_columns = [
    column
    for column in [
        "id",
        "name",
        "description",
        "format",
        "url",
        "created",
        "last_modified",
    ]
    if column in resources.columns
]

display(resources[display_columns])

def infer_year(resource):
    text = " ".join(
        str(resource.get(field, ""))
        for field in ["name", "description", "url"]
    )
    match = re.search(r"\b(20\d{2})\b", text)
    return int(match.group(1)) if match else pd.NA

resources["year"] = resources.apply(infer_year, axis=1)

# Ressources utilisables directement : ZIP ou CSV.
resources["format_normalized"] = (
    resources["format"]
    .fillna("")
    .astype(str)
    .str.lower()
    .str.strip()
)

resources["url_lower"] = resources["url"].astype(str).str.lower()

is_trip_file = (
    resources["format_normalized"].isin(["zip", "csv"])
    | resources["url_lower"].str.contains(r"\.(zip|csv)(\?|$)", regex=True)
)

selected_archives = (
    resources.loc[
        is_trip_file
        & resources["year"].isin(YEARS),
        ["year", "url", "name", "format", "id"]
    ]
    .dropna(subset=["url"])
    .sort_values(["year", "name"])
    .reset_index(drop=True)
)

if selected_archives.empty:
    print(
        "Aucune ressource annuelle ZIP/CSV n'a été identifiée automatiquement. "
        "Examinez le tableau complet `resources` affiché ci-dessus."
    )
else:
    print(f"Ressources historiques retenues : {len(selected_archives)}")
    display(selected_archives)

# Conserver un manifeste traçable pour l'analyse.
selected_archives.to_csv(
    PROCESSED_DIR / "bixi_archive_manifest_ckan.csv",
    index=False,
    encoding="utf-8-sig",
)

,id,name,description,format,url,created,last_modified
0,d0f48d14-880e-436f-96d3-bc354da5a9d2,Déplacements BIXI - par mois,(lien vers le site web de BIXI),CSV,https://bixi.com/fr/donnees-ouvertes/,2016-04-14T14:33:05.754751,None


Aucune ressource annuelle ZIP/CSV n'a été identifiée automatiquement. Examinez le tableau complet `resources` affiché ci-dessus.


/tmp/ipykernel_2873/2562962222.py:74: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  | resources["url_lower"].str.contains(r"\.(zip|csv)(\?|$)", regex=True)


## 5. Téléchargement, harmonisation et filtrage des trajets

In [13]:
def standardize_trip_columns(df):
    """
    Harmonise les colonnes BIXI dans un sous-ensemble standard.

    Colonnes retournées :
    - start_date
    - end_date
    - start_station_code
    - end_station_code
    - is_member
    - duration_sec
    """
    df = df.copy()

    df.columns = [
        normalize_column_name(column)
        for column in df.columns
    ]

    rename_map = {}

    for standard_name, aliases in COLUMN_ALIASES.items():
        source_column = next(
            (
                alias
                for alias in aliases
                if alias in df.columns
            ),
            None,
        )

        if source_column is not None:
            rename_map[source_column] = standard_name

    df = df.rename(columns=rename_map)

    required_columns = [
        "start_date",
        "end_date",
        "start_station_code",
        "end_station_code",
        "duration_sec",
    ]

    missing_columns = [
        column
        for column in required_columns
        if column not in df.columns
    ]

    if missing_columns:
        raise ValueError(
            "Colonnes nécessaires absentes : "
            f"{missing_columns}\n\n"
            f"Colonnes disponibles dans cette ressource :\n"
            f"{list(df.columns)}"
        )

    if "is_member" not in df.columns:
        df["is_member"] = pd.NA

    df["start_station_code"] = (
        df["start_station_code"]
        .astype("string")
        .str.strip()
    )

    df["end_station_code"] = (
        df["end_station_code"]
        .astype("string")
        .str.strip()
    )

    df["start_date"] = pd.to_datetime(
        df["start_date"],
        errors="coerce",
    )

    df["end_date"] = pd.to_datetime(
        df["end_date"],
        errors="coerce",
    )

    df["duration_sec"] = pd.to_numeric(
        df["duration_sec"],
        errors="coerce",
    )

    return df[
        [
            "start_date",
            "end_date",
            "start_station_code",
            "end_station_code",
            "is_member",
            "duration_sec",
        ]
    ]

## 6. Indicateurs par station, mois et paire origine-destination

In [15]:
# Réparation — recréer station_lookup pour Saint-Léonard

from pathlib import Path

import numpy as np
import geopandas as gpd

# Vérifier que les stations GBFS existent.
if "stations_current" not in globals():
    stations_file = (
        PROCESSED_DIR
        / "bixi_stations_current.parquet"
    )

    if not stations_file.exists():
        raise FileNotFoundError(
            "Le fichier des stations BIXI est absent. "
            "Réexécutez d'abord l'étape 2."
        )

    stations_current = pd.read_parquet(stations_file)

# Charger la limite filtrée générée à l'étape 3.
BOROUGH_FILE = Path(
    "/content/saint_leonard_limite.geojson"
)

if not BOROUGH_FILE.exists():
    raise FileNotFoundError(
        f"Limite introuvable : {BOROUGH_FILE}. "
        "Réexécutez d'abord l'étape 3."
    )

borough = gpd.read_file(BOROUGH_FILE)

if borough.crs is None:
    borough = borough.set_crs("EPSG:4326")
else:
    borough = borough.to_crs("EPSG:4326")

borough = borough[
    borough.geometry.notna()
].copy()

borough_union = borough.geometry.union_all()

# Transformer la table de stations BIXI en points géographiques.
stations_gdf = gpd.GeoDataFrame(
    stations_current,
    geometry=gpd.points_from_xy(
        stations_current["lon"],
        stations_current["lat"],
    ),
    crs="EPSG:4326",
)

# Le tampon doit être calculé dans une projection métrique.
BUFFER_METERS = 500

borough_metric = gpd.GeoSeries(
    [borough_union],
    crs="EPSG:4326",
).to_crs("EPSG:32188")

buffer_zone = (
    borough_metric
    .buffer(BUFFER_METERS)
    .to_crs("EPSG:4326")
    .iloc[0]
)

# Classer les stations : dans l'arrondissement ou dans la périphérie.
stations_gdf["in_saint_leonard"] = (
    stations_gdf.within(borough_union)
)

stations_gdf["within_buffer_saint_leonard"] = (
    stations_gdf.within(buffer_zone)
)

stations_stl = stations_gdf[
    stations_gdf["in_saint_leonard"]
    | stations_gdf["within_buffer_saint_leonard"]
].copy()

stations_stl["territory_class"] = np.where(
    stations_stl["in_saint_leonard"],
    "Saint-Léonard",
    f"Périphérie ≤ {BUFFER_METERS} m",
)

station_lookup = (
    stations_stl[
        [
            "station_code",
            "name",
            "lat",
            "lon",
            "capacity",
            "in_saint_leonard",
            "within_buffer_saint_leonard",
            "territory_class",
        ]
    ]
    .drop_duplicates("station_code")
    .copy()
)

station_lookup["station_code"] = (
    station_lookup["station_code"]
    .astype(str)
    .str.strip()
)

station_lookup.to_csv(
    PROCESSED_DIR / "bixi_stations_saint_leonard.csv",
    index=False,
    encoding="utf-8-sig",
)

print(f"Stations BIXI retenues : {len(station_lookup)}")
display(
    station_lookup.sort_values(
        ["territory_class", "name"]
    )
)

Stations BIXI retenues : 17


,station_code,name,lat,lon,capacity,in_saint_leonard,within_buffer_saint_leonard,territory_class
654,738,22e avenue / Everett,45.567038,-73.594269,19,False,True,Périphérie ≤ 500 m
434,484,26e avenue / Bélanger,45.567201,-73.585136,19,False,True,Périphérie ≤ 500 m
417,467,29e avenue / St-Zotique,45.567766,-73.579688,19,False,True,Périphérie ≤ 500 m
571,644,Carignan / Beaubien,45.585639,-73.561813,15,False,True,Périphérie ≤ 500 m
453,511,Centre ÉPIC (St-Zotique / 40e avenue),45.574147,-73.574621,23,False,True,Périphérie ≤ 500 m
727,825,Gare St-Michel / Montréal-Nord (Pie-IX / 56e),45.583198,-73.629787,15,False,True,Périphérie ≤ 500 m
685,774,Hôpital Santa Cabrini (de Pontoise / d'Évreux),45.581354,-73.572610,15,False,True,Périphérie ≤ 500 m
721,819,Parc Charleroi (Alfred / Louis-Francoeur),45.599535,-73.623603,15,False,True,Périphérie ≤ 500 m
723,821,Parc Maurice-Bélanger (Hébert / Denommée),45.587714,-73.631718,15,False,True,Périphérie ≤ 500 m
528,588,Michelet / Jean-Talon,45.576278,-73.583540,19,True,True,Saint-Léonard


## 7. Carte interactive de contrôle

In [17]:
import folium

borough_center = borough.to_crs("EPSG:4326").geometry.union_all().centroid
m = folium.Map(location=[borough_center.y, borough_center.x], zoom_start=13, tiles="CartoDB positron")

folium.GeoJson(borough.to_json(), name="Saint-Léonard").add_to(m)

for _, row in stations_stl.iterrows():
    color = "#1f77b4" if row["in_saint_leonard"] else "#ff7f0e"
    popup = (
        f"<b>{row['name']}</b><br>"
        f"Code : {row['station_code']}<br>"
        f"Territoire : {row['territory_class']}<br>"
        f"Capacité : {row.get('capacity', 'n.d.')}<br>"
        f"Vélos disponibles : {row.get('num_bikes_available', 'n.d.')}<br>"
        f"Bornes libres : {row.get('num_docks_available', 'n.d.')}"
    )
    folium.CircleMarker(
        location=[row["lat"], row["lon"]], radius=6, color=color, fill=True, fill_opacity=0.85, popup=popup
    ).add_to(m)

folium.LayerControl().add_to(m)
m

## Limites et notes

- Le flux GBFS `station_status` reflète un instantané au moment de l'exécution. Pour mesurer la disponibilité réelle (stations souvent vides ou pleines) dans le temps, prévoir une collecte planifiée récurrente (ex. toutes les 5-15 minutes) sur plusieurs semaines.
- La sélection territoriale dépend entièrement du fichier `saint_leonard_limite.geojson` fourni ; utiliser une source officielle plutôt qu'une approximation.
- Les résultats ne démontrent pas d'effet de rabattement vers les futures stations de la ligne bleue (non encore en service) ; ils établissent une ligne de base pour comparer avant/après leur mise en service.
- Mentionner l'attribution à BIXI Montréal (licence Creative Commons avec attribution) dans toute diffusion des résultats.